In [50]:
"""
CareLoop Clinics - No-Show Guard : PART 1 (Tasks 1.1 - 1.6, one script)
Run top to bottom (Colab cells in order, or: python part1_pipeline.py).
Inputs : appointments.csv, clinics.csv (also accepts appointment.csv / clinic.csv)
Outputs: rejected_rows.json, clinics_enriched.csv, appointments_clean.csv, load_data.sql
"""
import csv, json, os, glob
import numpy as np
import pandas as pd
import requests


In [49]:

# Locate input files

def find(*names):
    for d in [".", "data", "/content", "/content/data", "/mnt/user-data/uploads"]:
        for n in names:
            p = os.path.join(d, n)
            if os.path.exists(p):
                return p
    for n in names:
        hits = glob.glob(f"**/{n}", recursive=True) + glob.glob(f"/content/**/{n}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(
        f"Could not find any of {names}. Upload the CSVs to Colab (Files sidebar) "
        "or put them in a 'data' folder.")

APPT = find("appointments.csv", "appointment.csv")
CLIN = find("clinics.csv", "clinic.csv")
print("Using:", APPT, "|", CLIN)


Using: ./appointment.csv | ./clinic.csv


In [38]:

# TASK 1.1 - read with csv module only

print("=" * 60, "\nTASK 1.1")

def parse_row(row):
    row = dict(row)
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])
    return row

accepted, rejected = [], []
total = 0
with open(APPT, newline="") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)
    for row in reader:
        total += 1
        try:
            accepted.append(parse_row(row))
        except ValueError:
            rejected.append(row)
print("Rows read:", total, "| accepted:", len(accepted), "| rejected:", len(rejected))
with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)


TASK 1.1
Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows read: 1020 | accepted: 988 | rejected: 32


In [39]:

# TASK 1.2 - pandas cleaning, pass 1

print("=" * 60, "\nTASK 1.2")
df = pd.read_csv(APPT)
df.columns = df.columns.str.strip()
print("shape:", df.shape)
df.info()
print(df.isnull().sum())
print("duplicates:", df.duplicated().sum())
print("raw booking_channel spellings:", df["booking_channel"].nunique())

n0 = len(df)
df = df.drop_duplicates()                                             # (a)
print(f"(a) drop duplicates: {n0} -> {len(df)}")

df["booking_channel"] = df["booking_channel"].str.strip().str.title() # (b)
print("(b) booking_channel:", sorted(df["booking_channel"].dropna().unique()))

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")    # (c)
try:
    df["booked_on"] = pd.to_datetime(df["booked_on"])
except (ValueError, TypeError):
    df["booked_on"] = pd.to_datetime(df["booked_on"], format="mixed", errors="coerce")
print("(c) dates converted; unreadable appt_date:", df["appt_date"].isnull().sum())

n1 = len(df)
df = df.dropna(subset=["appt_date", "status"])                        # (d)
print(f"(d) drop missing appt_date/status: {n1} -> {len(df)}")



TASK 1.2
shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
appointment_id      0
patient_id          0
clinic_id           0

In [40]:

# TASK 1.3 - validity, gaps, outliers
print("=" * 60, "\nTASK 1.3")
df = df.copy()
for c in ["age", "distance_km", "fee_inr"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan          # (a)
print("missing age after invalid->NaN:", df["age"].isnull().sum())

med_age = df["age"].median()                                          # (b)
med_dist = df["distance_km"].median()
print("median age:", med_age, "| median distance:", med_dist)
df["age"] = df["age"].fillna(med_age).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(med_dist)

for col in ["distance_km", "fee_inr"]:                                # (c)
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    capped = ((df[col] < lo) | (df[col] > hi)).sum()
    df[col] = np.clip(df[col], lo, hi)
    print(f"{col}: Q1={q1}, Q3={q3}, lower={lo}, upper={hi}, capped={capped}")

df["distance_km"] = df["distance_km"].round(1)                        # (d)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("max fee after capping:", df["fee_inr"].max())


TASK 1.3
missing age after invalid->NaN: 17
median age: 48.0 | median distance: 8.25
distance_km: Q1=4.625, Q3=13.0, lower=-7.9375, upper=25.5625, capped=26
fee_inr: Q1=500.0, Q3=787.5, lower=68.75, upper=1218.75, capped=3
max fee after capping: 1219


In [42]:

# TASK 1.4 - features + NumPy summary

print("=" * 60, "\nTASK 1.4")
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)
print("lead_days range:", df["lead_days"].min(), "-", df["lead_days"].max())
print(f"no-shows: {df['no_show'].sum()} of {len(df)} ({df['no_show'].mean()*100:.1f}%)")

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("(a) column means:", np.round(X.mean(axis=0), 2))
mask = df["lead_days"].to_numpy() > 14
print(f"(b) lead_days>14: {mask.sum()} rows ({mask.mean()*100:.1f}%)")
x = df["distance_km"].to_numpy()
z = (x - np.mean(x)) / np.std(x)
print("(c) |z|>2 for distance_km:", (np.abs(z) > 2).sum())


TASK 1.4
lead_days range: 1 - 45
no-shows: 218 of 990 (22.0%)
(a) column means: [ 48.17   9.5  635.71  11.82]
(b) lead_days>14: 296 rows (29.9%)
(c) |z|>2 for distance_km: 53


In [48]:

# TASK 1.5 - enrich clinics from Open-Meteo

print("=" * 60, "\nTASK 1.5")
clinics = pd.read_csv(CLIN)
URL = "https://api.open-meteo.com/v1/forecast"

def get_temp(lat, lon):
    """Return (temp, reason). One attempt; the caller retries once."""
    try:
        r = requests.get(URL, timeout=10, params={
            "latitude": lat, "longitude": lon,
            "hourly": "temperature_2m", "forecast_days": 1})
        if r.status_code != 200:
            return None, f"status {r.status_code}"
        data = r.json()
        if "hourly" not in data:
            return None, "hourly missing"
        return data["hourly"]["temperature_2m"][0], None
    except requests.exceptions.RequestException as e:
        return None, f"request error: {e.__class__.__name__}"

temps = []
for _, c in clinics.iterrows():
    t, why = get_temp(c["latitude"], c["longitude"])
    if t is None:                                   # retry once
        t, why = get_temp(c["latitude"], c["longitude"])
    if t is None:
        print(f"FAILED: {c['clinic_name']} -> {why}")
    temps.append(t)
clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print(clinics[["clinic_id", "clinic_name", "temp_c"]])


TASK 1.5
  clinic_id               clinic_name  temp_c
0       C01     Kothrud Family Clinic    14.4
1       C02      Hadapsar Care Centre    15.7
2       C03    Sitabuldi Health Point    22.3
3       C04      Malviya Nagar Clinic    15.3
4       C05     Vaishali Nagar Clinic    15.3
5       C06  Edappally Medical Centre    24.2
6       C07        Vijay Nagar Clinic    15.2
7       C08    Gomti Nagar Care Point    17.0


In [47]:

# TASK 1.6 - export clean table + SQL load script

print("=" * 60, "\nTASK 1.6")
COLS = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
        "age", "gender", "distance_km", "fee_inr", "booking_channel",
        "visit_type", "reminder_sent", "prior_no_shows", "status",
        "lead_days", "no_show"]

df = df.sort_values("appointment_id").copy()
# pd.to_datetime first, so re-running this cell does not fail
df["booked_on"] = pd.to_datetime(df["booked_on"]).dt.strftime("%Y-%m-%d")
df["appt_date"] = pd.to_datetime(df["appt_date"]).dt.strftime("%Y-%m-%d")
out = df[COLS]

print("appointment_id unique:", out["appointment_id"].is_unique,
      "| distinct:", out["appointment_id"].nunique())
print("missing values per column:\n", out.isnull().sum().to_string())
assert out["appointment_id"].is_unique and out.isnull().sum().sum() == 0
out.to_csv("appointments_clean.csv", index=False)
print("appointments_clean.csv:", out.shape)

def sql_val(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "NULL"
    if isinstance(v, str):
        return f"'{v}'"
    return str(v)

n_ins = 0
with open("load_data.sql", "w") as f:
    for _, c in clinics.iterrows():
        vals = [sql_val(c[k]) for k in clinics.columns]
        f.write(f"INSERT INTO clinics VALUES ({', '.join(vals)});\n")
        n_ins += 1
    for row in out.itertuples(index=False):
        vals = [sql_val(v.item() if hasattr(v, "item") else v) for v in row]
        f.write(f"INSERT INTO appointments VALUES ({', '.join(vals)});\n")
        n_ins += 1
print("INSERT statements written:", n_ins)

TASK 1.6
appointment_id unique: True | distinct: 990
missing values per column:
 appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
appointments_clean.csv: (990, 16)


NameError: name 'clinics' is not defined